opening for positional players:  https://lichess.org/study/0vNGLxJw/smynh89h   
opening dataset:https://huggingface.co/datasets/Lichess/chess-openings   

In [12]:
from huggingface_hub import notebook_login
from datasets import load_dataset
import pandas as pd
import numpy as np
import sqlite3
notebook_login()
#在输入中输入token

In [13]:
dataset=load_dataset("Lichess/chess-openings")
print(dataset['train'].features)
dataset.set_format(type="python", columns=["eco", "name", "pgn", "uci", "epd"])
openings_df = pd.DataFrame(dataset["train"]) 

{'eco-volume': Value('string'), 'eco': Value('string'), 'name': Value('string'), 'img': Image(mode=None, decode=True), 'pgn': Value('string'), 'uci': Value('string'), 'epd': Value('string')}


In [14]:
class TrieNode:
    def __init__(self):
        self.children={}
        self.opening_name=None

def build_tree(openings_df):
    root=TrieNode()
    for _,row in openings_df.iterrows():
        moves=row['uci'].split()
        node=root
        for move in moves:
            if move not in node.children:
                node.children[move]=TrieNode()
            node=node.children[move]
        node.opening_name=row['name']
    return root

def openings_identify(game_moves,trie_root):
    if not game_moves:
        return "Unknown"

    node=trie_root
    game_moves=game_moves.replace(',',' ')
    for move in game_moves.split():
        if move in node.children:
            node=node.children[move]
            if node.opening_name:
                return node.opening_name
        else:
            break
    return 'Unknown'
trie_root = build_tree(openings_df)
print(list(trie_root.children.keys())[:10])
db_file = r"C:\sqlite3\chess.db"
conn=sqlite3.connect(db_file)
games_df=pd.read_sql('SELECT * FROM games',conn)
tire_root=build_tree(openings_df)
games_df['Opening'] = games_df['Moves'].apply(lambda x: openings_identify(x, tire_root))
#映射字典
name_to_eco=dict(zip(openings_df['name'],openings_df['eco']))
games_df['ECO']=games_df['Opening'].map(name_to_eco)
print(games_df[['Moves', 'Opening','ECO']].head(50))
print(list(tire_root.children.keys())[:10])

['g1h3', 'e2e3', 'a2a3', 'f2f3', 'h2h3', 'g2g4', 'g2g3', 'h2h4', 'd2d3', 'b2b4']
                                                Moves            Opening  ECO
0   d2d4,g8f6,c2c4,e7e6,g1f3,b7b6,g2g3,c8b7,f1g2,f...  Queen's Pawn Game  D00
1   d2d4,d7d5,b1c3,c7c6,c1f4,g8f6,e2e3,c8f5,g1f3,e...  Queen's Pawn Game  D00
2   d2d4,g8f6,c2c4,c7c5,d4d5,b7b5,c4b5,a7a6,b5a6,e...  Queen's Pawn Game  D00
3   g1f3,d7d5,e2e3,b8c6,d2d4,c8f5,c2c4,e7e6,a2a3,a...  Zukertort Opening  A06
4   d2d4,g8f6,c2c4,e7e6,g2g3,f8b4,b1d2,c7c5,a2a3,b...  Queen's Pawn Game  D00
5   d2d4,d7d5,c2c4,c7c6,g1f3,g8f6,b1c3,d5c4,a2a4,c...  Queen's Pawn Game  D00
6                                                None            Unknown  NaN
7   e2e4,e7e6,d2d4,d7d5,b1d2,d5e4,d2e4,f8e7,g1f3,g...   King's Pawn Game  C20
8   e2e4,c7c5,b1c3,b8c6,g2g3,e7e6,f1g2,g7g6,d2d3,f...   King's Pawn Game  C20
9   e2e4,c7c5,g1f3,e7e6,g2g3,b8c6,f1g2,g8f6,d1e2,d...   King's Pawn Game  C20
10  e2e4,c7c5,g1f3,e7e6,d2d4,c5d4,f3d4,g8f6,b1c3,b...   King'

In [17]:
games_df.shape
games_df.head()

,uid,Event,Site,Date,Round,White,Black,Result,WhiteElo,BlackElo,TimeControl,EndTime,Termination,Moves,Opening,ECO
0,1,Early-Titled-Tuesday-Blitz-April-01-2025,Chess.com,2025.04.01,1,Petra Kejzar,Daniel Barria,0-1,2177,2637,180+1,15:01:06 GMT+0000,manitodeplomo胜，对手认输,"d2d4,g8f6,c2c4,e7e6,g1f3,b7b6,g2g3,c8b7,f1g2,f...",Queen's Pawn Game,D00
1,2,Early-Titled-Tuesday-Blitz-April-01-2025,Chess.com,2025.04.01,1,Nataliya Buksa,Jorge Carlos Antonio,1-0,2639,2195,180+1,15:02:10 GMT+0000,Natalya_Buksa胜，对手认输,"d2d4,d7d5,b1c3,c7c6,c1f4,g8f6,e2e3,c8f5,g1f3,e...",Queen's Pawn Game,D00
2,3,Early-Titled-Tuesday-Blitz-April-01-2025,Chess.com,2025.04.01,1,Hubert Zieba,Nikolaos Skiadopoulos,1-0,2607,2049,180+1,15:02:41 GMT+0000,Chomiczek786胜，对手认输,"d2d4,g8f6,c2c4,c7c5,d4d5,b7b5,c4b5,a7a6,b5a6,e...",Queen's Pawn Game,D00
3,4,Early-Titled-Tuesday-Blitz-April-01-2025,Chess.com,2025.04.01,1,Karina Ambartsumova,Nino Maisuradze,1-0,2709,2358,180+1,15:02:42 GMT+0000,karinachess1胜，对手认输,"g1f3,d7d5,e2e3,b8c6,d2d4,c8f5,c2c4,e7e6,a2a3,a...",Zukertort Opening,A06
4,5,Early-Titled-Tuesday-Blitz-April-01-2025,Chess.com,2025.04.01,1,Alexander Donchenko,Andriy Diachek,1-0,2835,2494,180+1,15:02:49 GMT+0000,Alexander_Donchenko胜，对手认输,"d2d4,g8f6,c2c4,e7e6,g2g3,f8b4,b1d2,c7c5,a2a3,b...",Queen's Pawn Game,D00
